# Project 3: Human Evaluation of a Simple RAG

The application retrieves fable passages and generates an answer. Jack, Jane,
and Jill independently judge the **same saved answers** with a 1-5 helpfulness
rubric. Python combines their ratings; it does not supply human judgments.

**Prepare first**, from the repository root:

```powershell
uv run python 3_fables_human_eval/create_vectordb.py
```

Select the root `.venv` kernel in VS Code. Run cells individually from the
repository root or this project folder. Restart the kernel when switching from
another project's `models`/`tools` modules or after editing imported helpers.
Preparation, reporting, and rating collection are separate from this notebook.


## 1. Set the project path
This makes the project's helper modules importable.


In [1]:
import json
import sys
from pathlib import Path

project_root = Path.cwd()
if project_root.name != "3_fables_human_eval":
    project_root = project_root / "3_fables_human_eval"
assert project_root.is_dir(), "Run from the repository root or Project 3 folder."
sys.path.insert(0, str(project_root))


## 2. Open the existing store and connect the model

The model configuration is in `models/config.py`. Groq reads `GROQ_API_KEY`
from the shared root `.env`. Opening Chroma does not rebuild the index.
The prompt permits practical advice but grounds story facts in retrieved text.


In [2]:
from langchain_core.prompts import ChatPromptTemplate
from models.config import DEFAULT_K
from models.vector_store import load_vector_store
from models.llm import load_model

k = DEFAULT_K
vector_store = load_vector_store()
retriever = vector_store.as_retriever(search_kwargs={"k": k})
model = load_model()

prompt_text = (project_root / "prompts/answer.txt").read_text(encoding="utf-8")
prompt = ChatPromptTemplate.from_messages([
    ("system", prompt_text),
    ("human", "{question}"),
])


c:\a\pproj\AI\zzTwo\Evals2\core\eval-genius\.venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
Loading weights: 100%|██████████| 199/199 [00:00<00:00, 2487.64it/s]


## 3. Create the simple RAG

Retrieve passages, format the context, and invoke the answer model.
Each question is independent. Reference `fable_ids` and human ratings never
guide retrieval: only the question is sent to the retriever.

Retain the exact retrieved passages with the answer so a poor response can
later be traced to missing context or generation.


In [3]:
def ask_rag(question):
    documents = retriever.invoke(question)
    context = "\n\n".join(
        f"[page {doc.metadata['page']}]\n{doc.page_content}"
        for doc in documents
    )
    messages = prompt.format_messages(context=context, question=question)
    answer = model.invoke(messages).content
    return {
        "answer": answer,
        "retrieved_context": [
            {"text": doc.page_content, **doc.metadata}
            for doc in documents
        ],
    }


## 4. Load questions and review the rubric

Questions and rubric are AI-prepared drafts. Read them and the supplied fable
versions before collecting final human judgments. Scores reflect the whole RAG,
including any retrieval failures.


In [4]:
from tools.results import read_json
from tools.responses import generate_responses

questions = read_json(project_root / "docs/goldends/questions.json")
rubric = read_json(project_root / "docs/goldends/rubric.json")

print(f"Questions: {len(questions)}")
for anchor in rubric["anchors"]:
    print(f"{anchor['score']}: {anchor['description']}")


Questions: 15
1: Unhelpful: irrelevant, substantially wrong, or invents the central requested fact.
2: Mostly unhelpful: addresses a small part but has major gaps, confusion, or errors.
3: Partly helpful: addresses the main idea but misses an important requested part or explanation.
4: Helpful: accurate, clear, and addresses the request with only a minor omission or weakness.
5: Fully helpful: accurate to the supplied text, clear, and sufficiently complete for the request. A concise acknowledgment that a fact is missing can earn 5.


## 5. Generate a response run — optional when using saved answers

This cell makes 15 model calls and saves a new immutable response file when
the complete loop succeeds. It records the prompt, model, retrieval settings,
rubric, source text for review, and the returned passages.

Run this cell for a **new experiment**. Skip it to evaluate an existing response
run. Rerunning it creates a new run; the old responses and ratings are preserved.
If any call fails, stop and inspect it; no new complete response file is saved.


In [5]:
response_path = generate_responses(
    questions, ask_rag, k=k, prompt_text=prompt_text, label="baseline"
)
print(f"Saved response run: {response_path.name}")


Answered h001
Answered h002
Answered h003
Answered h004
Answered h005
Answered h006
Answered h007
Answered h008
Answered h009
Answered h010
Answered h011
Answered h012
Answered h013
Answered h014
Answered h015
Saved response run: responses_20261014_215116_945329.json


## 6. Select a saved response run

This cell lists saved runs and selects the latest one. To choose an older run,
replace the last line assigning `response_path` with its explicit path.
Always confirm the printed run ID before rating or scoring it.

**Offline route:** to use saved answers without loading models, run only the
path cell (step 1) and continue here. No API key or vector store is required.


In [6]:
from tools.results import read_json, print_report, save_report
from tools.ratings import prepare_ratings, rating_progress
from tools.evaluation import combine_scores

saved_runs = sorted((project_root / "docs/evaluatedds").glob("responses_*.json"))
assert saved_runs, "Generate a response run first."
for path in saved_runs:
    print(path.name)

response_path = saved_runs[-1]
run = read_json(response_path)
print(f"\nSelected: {run['response_run_id']} | model: {run['model']} | k={run['k']}")


responses_20261014_005400_839793.json
responses_20261014_215116_945329.json

Selected: responses_20261014_215116_945329 | model: openai/gpt-oss-120b | k=5


## 7. Prepare independent rating sheets

Open `rate_answers.ipynb` for the three reviewers. Give each person the selected
response filename. Each should rate independently using the same frozen rubric,
without looking at the other reviewers' scores.

The files live in `docs/ratings/<response_run_id>/`. Existing sheets are not reset.
Each reviewer provides one integer score and one brief reason for all 15 answers.
The progress display reveals only counts.


In [8]:
rating_paths = prepare_ratings(run)
for reviewer, completed in rating_progress(run).items():
    print(f"{reviewer}: {completed}/{len(run['responses'])}")


Jack: 5/15
Jane: 5/15
Jill: 5/15


## 8. Combine completed ratings

**Run only after Jack, Jane, and Jill have all finished.**
The helper rejects missing or invalid scores, missing reasons, duplicate or
mismatched question IDs, different response runs, and different rubric versions.

For each question: `(Jack + Jane + Jill) / 3`.
Overall: average the 15 question means. Everyone has equal weight.
Report helpfulness out of 5, not as an accuracy percentage.


In [10]:
reviews = [read_json(path) for path in prepare_ratings(run)]
report = combine_scores(run, reviews)
print_report(report)
report_path = save_report(report)
print(f"\nReport saved: {report_path.name}")


Response run: responses_20261014_215116_945329
Questions: 15 | Ratings: 45
Combined helpfulness: 3.82 / 5
Jack: 5.00 / 5
Jane: 3.60 / 5
Jill: 2.87 / 5

Question | Jack | Jane | Jill | Mean | Spread
h001 | 5 | 3 | 4 | 4.00 | 2
h002 | 5 | 4 | 3 | 4.00 | 2
h003 | 5 | 5 | 3 | 4.33 | 2
h004 | 5 | 3 | 4 | 4.00 | 2
h005 | 5 | 4 | 4 | 4.33 | 1
h006 | 5 | 4 | 2 | 3.67 | 3
h007 | 5 | 4 | 3 | 4.00 | 2
h008 | 5 | 3 | 2 | 3.33 | 3
h009 | 5 | 5 | 3 | 4.33 | 2
h010 | 5 | 2 | 2 | 3.00 | 3
h011 | 5 | 3 | 1 | 3.00 | 4
h012 | 5 | 1 | 3 | 3.00 | 4
h013 | 5 | 3 | 4 | 4.00 | 2
h014 | 5 | 5 | 3 | 4.33 | 2
h015 | 5 | 5 | 2 | 4.00 | 3

Questions to discuss: h001, h002, h003, h004, h006, h007, h008, h009, h010, h011, h012, h013, h014, h015

Report saved: combined_20261014_220526_658994.json


## 9. Inspect disagreements and plan one improvement

Read the saved reasons for questions with low means or a spread of 2 or more.
For example, scores 1, 3, and 5 average to 3 but represent substantial disagreement.

Distinguish missing retrieved evidence from a poor explanation. Change one setting
or prompt instruction, create a new response run, and collect fresh independent
ratings. Keep questions and rubric fixed for a fair comparison.

A working implementation is ready; final human scores require the reviewers'
actual judgments. Do not replace them with model-generated or invented ratings.
